# Análise de demanda BebiBem
Este notebook reproduz a análise da loja própria usando as funções públicas de `bebibem`. A unidade de medida é **unidades vendidas**.

## Dados e notação
A observação $y_t$ é a demanda no mês $t$; $n$ é o número de observações e $x$ é o índice mensal, com $x=0$ em 2022-01. O loader aceita os layouts wide e long e normaliza ambos para a tabela tidy.

O eixo X de toda figura de série temporal usa o mesmo índice, agora formalizado como o inteiro contínuo
$$T = 12\,(\text{ano} - 2022) + (\text{mês} - 1),$$
com $T=0$ em janeiro/2022, $T=35$ em dezembro/2024 (fim do histórico) e $T=36$ em janeiro/2025 (início da previsão), nunca a data de calendário no eixo.

In [1]:
import os
from functools import partial
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from numpy.polynomial import Polynomial

from bebibem.clustering import cluster_months, cluster_series, seasonal_index_matrix
from bebibem.features import build_features, next_row, split_train_holdout
from bebibem.figures import (
    fig_backtest,
    fig_error_table,
    fig_forecast_2025_bar,
    fig_forecast_2025_curves_all,
    fig_forecast_2025_total_bar,
    fig_forecast_all,
    fig_historical_bar,
    fig_forecast_product,
    fig_interpolation,
    fig_metrics_bar,
    fig_month_clusters,
    fig_moving_average,
    fig_points_all,
    fig_points_product,
    fig_series_clusters,
)
from bebibem.forecast import fit_final, reconstruct_winning_series
from bebibem.interpolation import (
    fill_gaps_linear,
    polynomial_fit,
)
from bebibem.io_data import load_demand
from bebibem.models import build_estimators, recursive_backtest
from bebibem.pipeline import PipelineConfig, run
from bebibem.plotting import save_figure
from bebibem.regression import (
    exponential_regression_loglinear,
    exponential_regression_normal_equation,
    linear_regression_normal_equation,
)
from bebibem.smoothing import moving_average_table

project_root = Path.cwd()
example_path = project_root / "data/example/demanda_exemplo.csv"
#example_path = project_root / "data/raw/demanda_loja_propria.csv"
if example_path.is_file() is False:
    project_root = project_root.parent
input_path = project_root / "data/raw/demanda_loja_propria.csv"
#example_path = project_root / "data/raw/demanda_loja_propria.csv"
notebook_output = os.environ.get("BEBIBEM_NOTEBOOK_OUTDIR")
reports_path = (
    Path(notebook_output)
    if notebook_output is not None
    else project_root / "reports/notebook-example"
)
frame = load_demand(input_path)
products = frame["produto"].drop_duplicates().tolist()
display(frame.head())

,produto,produto_label,ano,mes,data,unidades
0,doce_de_frutas,Doce de frutas,2022,1,2022-01-01,63.0
1,doce_de_frutas,Doce de frutas,2022,2,2022-02-01,34.0
2,doce_de_frutas,Doce de frutas,2022,3,2022-03-01,14.0
3,doce_de_frutas,Doce de frutas,2022,4,2022-04-01,22.0
4,doce_de_frutas,Doce de frutas,2022,5,2022-05-01,63.0


## Pontos observados
O gráfico mostra $y_t$ observado ao longo do tempo; meses sem observação permanecem sem marcador. Símbolos: $y_t$ = unidades observadas e $t$ = mês calendário.

In [2]:
point_figures = {product: fig_points_product(frame, product) for product in products}
display(fig_points_all(frame))
for figure in point_figures.values():
    display(figure)
    plt.close(figure)

<Figure size 1000x600 with 1 Axes>

<Figure size 1000x600 with 1 Axes>

<Figure size 1000x600 with 1 Axes>

<Figure size 1000x600 with 1 Axes>

## Interpolações linear e polinomial
Entre meses observados, usamos
$$\hat{y}(t)=y_i+\frac{y_{i+1}-y_i}{t_{i+1}-t_i}(t-t_i).$$
O ajuste polinomial é $\hat{y}(x)=\sum_{j=0}^{d}a_jx^j$, com $d$ escolhido por RMSE de holdout (ou fixado pelo usuário). Aqui $x=0$ em 2022-01, $d$ é o grau e $\hat{y}_t$ é o previsto.

$$ \hat{y}(t) = y_i + \frac{y_{i+1}-y_i}{t_{i+1}-t_i}\,(t-t_i), \qquad t \in [t_i,\, t_{i+1}] $$

$$ \hat{y}(x)=\sum_{j=0}^{d} a_j\,x^{j}, \qquad \mathbf{a}=\arg\min_{\mathbf{a}}\sum_{t=1}^{n}\left(y_t-\sum_{j=0}^{d}a_j\,x_t^{j}\right)^{2} $$

In [3]:
analysis_frame = frame.copy()
for product in products:
    product_mask = analysis_frame["produto"].eq(product)
    product_series = (
        analysis_frame.loc[product_mask].sort_values("data").set_index("data")["unidades"]
    )
    filled_series = fill_gaps_linear(product_series)
    analysis_frame.loc[product_mask, "unidades"] = filled_series.to_numpy()
polynomial_fits = {
    product: polynomial_fit(
        analysis_frame.loc[analysis_frame["produto"].eq(product)]
        .sort_values("data")
        .index.to_series()
        .reset_index(drop=True)
        .index.to_numpy(dtype=float),
        analysis_frame.loc[analysis_frame["produto"].eq(product)]
        .sort_values("data")["unidades"]
        .to_numpy(dtype=float),
    )
    for product in products
}
interpolation_figures = {product: fig_interpolation(frame, product) for product in products}
for figure in interpolation_figures.values():
    display(figure)
    plt.close(figure)
candidate_table = pd.concat(
    {product: fit.candidates for product, fit in polynomial_fits.items()}, names=["produto"]
).reset_index()
display(candidate_table)

<Figure size 1000x600 with 1 Axes>

<Figure size 1000x600 with 1 Axes>

<Figure size 1000x600 with 1 Axes>

,produto,level_1,degree,rmse
0,doce_de_frutas,0,1,25.460479
1,doce_de_frutas,1,2,25.417265
2,doce_de_frutas,2,3,54.273449
3,doce_de_frutas,3,4,806.534652
4,doce_de_frutas,4,5,1849.937315
5,geleia_de_frutas,0,1,9.353280
6,geleia_de_frutas,1,2,18.798217
7,geleia_de_frutas,2,3,15.815570
8,geleia_de_frutas,3,4,317.038398
9,geleia_de_frutas,4,5,950.632950


## Extensão: grau polinomial de 1 a 20 com validação temporal
A seleção de produção (`polynomial_fit`) restringe o grau a $d\in\{1,\dots,5\}$ por decisão anti-overfitting documentada em `.specs/features/previsao-demanda-bebibem/spec.md` ("grau alto passa pelos pontos e explode fora deles"). Para evidenciar esse comportamento, esta seção estende a varredura até $d=20$ usando a mesma validação **temporal** do restante do notebook: treino em 2022–2023, holdout em 2024 — a mesma partição do backtest de ML (DEM-09). Para cada grau $d$ ajustamos

$$ \hat{y}(x; d)=\sum_{j=0}^{d} a_j\,x^{j} $$

sobre o treino e avaliamos no holdout

$$ \mathrm{RMSE}(d)=\sqrt{\frac{1}{|T_0|}\sum_{t\in T_0}\left(y_t-\hat{y}(x_t; d)\right)^{2}}, \qquad T_0=\{t : \mathrm{ano}(t)=2024\}. $$

Como antes, $x$ é o índice mensal com $x=0$ em 2022-01. O gráfico de $\mathrm{RMSE}(d)$ por grau e as curvas sobrepostas mostram o fenômeno de Runge: o erro de treino cai, mas o erro de holdout piora a partir de um certo $d$ — é exatamente o motivo pelo qual a produção fixa $d\le 5$. Cada ajuste (produto × grau) também é gravado individualmente em PNG/SVG/PDF via `bebibem.plotting.save_figure`, em `reports/.../figures/{png,svg,pdf}/regressao_polinomial/grau_<dd>_<produto>.<formato>`.

In [4]:
extended_degrees = list(range(1, 21))
holdout_year = 2024
extended_rows = []
extended_fits = {}
for product in products:
    product_frame = (
        analysis_frame.loc[analysis_frame["produto"].eq(product)]
        .sort_values("data")
        .reset_index(drop=True)
    )
    x_all = product_frame.index.to_numpy(dtype=float)
    y_all = product_frame["unidades"].to_numpy(dtype=float)
    year_values = product_frame["ano"].to_numpy()
    train_mask = year_values < holdout_year
    holdout_mask = year_values == holdout_year
    x_train, y_train = x_all[train_mask], y_all[train_mask]
    x_holdout, y_holdout = x_all[holdout_mask], y_all[holdout_mask]
    product_fits = {}
    for degree in extended_degrees:
        if len(x_train) <= degree:
            continue
        fitted = Polynomial.fit(x_train, y_train, deg=degree).convert()
        prediction_train = fitted(x_train)
        residual_train = y_train - prediction_train
        total_train = float(np.sum((y_train - y_train.mean()) ** 2))
        r2_train = 1.0 if total_train == 0 else float(1 - np.sum(residual_train**2) / total_train)
        prediction_holdout = fitted(x_holdout)
        rmse_holdout = float(np.sqrt(np.mean((y_holdout - prediction_holdout) ** 2)))
        extended_rows.append(
            {
                "produto": product,
                "grau": degree,
                "r2_treino": r2_train,
                "rmse_holdout_2024": rmse_holdout,
            }
        )
        product_fits[degree] = fitted
    extended_fits[product] = product_fits

extended_degree_table = pd.DataFrame(extended_rows)
display(extended_degree_table)

figure, axis = plt.subplots(figsize=(10, 6))
for product in products:
    subset = extended_degree_table.loc[extended_degree_table["produto"].eq(product)]
    axis.plot(subset["grau"], subset["rmse_holdout_2024"], marker="o", label=product)
axis.set_xlabel("grau do polinômio (d)")
axis.set_ylabel("RMSE no holdout 2024")
axis.set_title("RMSE por grau polinomial (1-20) - validacao temporal 2022-2023 / 2024")
axis.set_xticks(extended_degrees)
axis.tick_params(axis="x", labelrotation=90)
axis.set_yscale("log")
axis.legend()
axis.grid(alpha=0.3, which="both")
display(figure)
plt.close(figure)

figures_dir = reports_path / "figures"
saved_degree_plots: list[Path] = []
display_degrees = {1, 2, 3, 5, 10, 15, 20}
for product in products:
    product_frame = (
        analysis_frame.loc[analysis_frame["produto"].eq(product)]
        .sort_values("data")
        .reset_index(drop=True)
    )
    x_all = product_frame.index.to_numpy(dtype=float)
    y_all = product_frame["unidades"].to_numpy(dtype=float)
    year_values = product_frame["ano"].to_numpy()
    dense_x = np.linspace(x_all.min(), x_all.max(), 400)
    holdout_x_values = x_all[year_values == holdout_year]
    for degree in extended_degrees:
        fitted = extended_fits[product].get(degree)
        if fitted is None:
            continue
        metrics_row = extended_degree_table.loc[
            extended_degree_table["produto"].eq(product) & extended_degree_table["grau"].eq(degree)
        ].iloc[0]
        figure, axis = plt.subplots(figsize=(10, 6))
        axis.scatter(x_all, y_all, color="black", zorder=5, label="observado")
        axis.plot(
            dense_x,
            fitted(dense_x),
            label=f"grau {degree} (R2 treino = {metrics_row['r2_treino']:.3f})",
        )
        axis.axvspan(
            holdout_x_values.min(), x_all.max(), color="grey", alpha=0.12, label="holdout 2024"
        )
        axis.set_ylim(min(0.0, float(y_all.min()) * 1.2), float(y_all.max()) * 1.8)
        axis.set_xlabel("x (meses desde 2022-01)")
        axis.set_ylabel("unidades")
        axis.set_title(
            f"Ajuste polinomial grau {degree} - {product} "
            f"(RMSE holdout 2024 = {metrics_row['rmse_holdout_2024']:.2f})"
        )
        axis.legend()
        figure_name = f"regressao_polinomial/grau_{degree:02d}_{product}"
        saved_degree_plots.extend(save_figure(figure, figure_name, figures_dir))
        if degree in display_degrees:
            display(figure)
        plt.close(figure)

print(f"{len(saved_degree_plots)} arquivos gravados em {figures_dir / 'png' / 'regressao_polinomial'}")

,produto,grau,r2_treino,rmse_holdout_2024
0,doce_de_frutas,1,0.001762,2.546048e+01
1,doce_de_frutas,2,0.004376,2.541726e+01
2,doce_de_frutas,3,0.017256,5.427345e+01
3,doce_de_frutas,4,0.357242,8.065347e+02
4,doce_de_frutas,5,0.400533,1.849937e+03
5,doce_de_frutas,6,0.401766,1.233841e+03
6,doce_de_frutas,7,0.478229,1.665215e+04
7,doce_de_frutas,8,0.537228,4.173479e+04
8,doce_de_frutas,9,0.548748,5.691377e+04
9,doce_de_frutas,10,0.573820,6.284754e+05


<Figure size 1000x600 with 1 Axes>

<Figure size 1000x600 with 1 Axes>

<Figure size 1000x600 with 1 Axes>

<Figure size 1000x600 with 1 Axes>

...

<Figure size 1000x600 with 1 Axes>

<Figure size 1000x600 with 1 Axes>

<Figure size 1000x600 with 1 Axes>

<Figure size 1000x600 with 1 Axes>

180 arquivos gravados em /Users/borinmacedo/GitRepos/insper-bebidem/reports/notebook-example/figures/png/regressao_polinomial


## Médias móveis
A média móvel trailing de janela $w$ é
$$MA_w(t)=\frac{1}{w}\sum_{i=0}^{w-1}y_{t-i}.$$
Para uma janela par centrada, usamos a média dupla correspondente a $MA^c_w(t)$. Aqui $w\in\{3,12\}$ e $y_t$ é a série preenchida.

$$ MA_w(t)=\frac{1}{w}\sum_{i=0}^{w-1} y_{t-i} $$

$$ MA^{c}_{w}(t)=\frac{1}{2}\left(\frac{1}{w}\sum_{i=-w/2}^{w/2-1} y_{t+i}+\frac{1}{w}\sum_{i=-w/2+1}^{w/2} y_{t+i}\right) $$

In [5]:
moving_table = moving_average_table(analysis_frame, windows=(3, 12))
moving_figures = {product: fig_moving_average(analysis_frame, product) for product in products}
for figure in moving_figures.values():
    display(figure)
    plt.close(figure)
display(moving_table.head())

<Figure size 1000x600 with 1 Axes>

<Figure size 1000x600 with 1 Axes>

<Figure size 1000x600 with 1 Axes>

,produto,produto_label,ano,mes,data,unidades,ma_3,ma_12,ma_centered_12
0,doce_de_frutas,Doce de frutas,2022,1,2022-01-01,63.0,NaN,NaN,NaN
1,doce_de_frutas,Doce de frutas,2022,2,2022-02-01,34.0,NaN,NaN,NaN
2,doce_de_frutas,Doce de frutas,2022,3,2022-03-01,14.0,37.000000,NaN,NaN
3,doce_de_frutas,Doce de frutas,2022,4,2022-04-01,22.0,23.333333,NaN,NaN
4,doce_de_frutas,Doce de frutas,2022,5,2022-05-01,63.0,33.000000,NaN,NaN


## Clusters sazonais
O índice sazonal mensal é $S_m=\bar{y}_m/\bar{y}$. O KMeans minimiza
$$\min_{C_1,\ldots,C_k}\sum_{j=1}^{k}\sum_{x\in C_j}\lVert x-\mu_j\rVert_2^2,$$
e escolhemos $k$ pelo maior silhouette $s(i)=(b(i)-a(i))/\max\{a(i),b(i)\}$. Símbolos: $m$ = mês, $k$ = clusters e $x$ = vetor de índices.

$$ S_m=\frac{\bar{y}_m}{\bar{y}}, \qquad \bar{y}_m=\frac{1}{|A|}\sum_{a\in A} y_{a,m} $$

$$ \min_{C_1,\dots,C_k}\ \sum_{j=1}^{k}\sum_{x\in C_j}\lVert x-\mu_j\rVert_2^{2}, \qquad \mu_j=\frac{1}{|C_j|}\sum_{x\in C_j}x $$

$$ s(i)=\frac{b(i)-a(i)}{\max\{a(i),\,b(i)\}} $$

In [6]:
seasonal_matrix = seasonal_index_matrix(analysis_frame)
month_clusters = cluster_months(seasonal_matrix, seed=42)
series_clusters = cluster_series(analysis_frame, seed=42)
display(fig_month_clusters(seasonal_matrix, month_clusters))
display(fig_series_clusters(analysis_frame, series_clusters))

<Figure size 1000x600 with 1 Axes>

<Figure size 1000x600 with 1 Axes>

## Comparação dos modelos
A matriz supervisionada usa $X_t=[y_{t-1},y_{t-12},\mathbb{1}_{m=2..12},\mathrm{trend},\mathbb{1}_{c_m}]$ e o Ridge resolve
$$\min_{\beta}\lVert y-X\beta\rVert_2^2+\alpha\lVert\beta\rVert_2^2.$$
Comparamos Ridge, Random Forest, Gradient Boosting e as regressões clássicas. As cinco métricas são $MAD$, $MAPE$, $MSE$, $RMSE$ e $MPE$; $n$ é o tamanho do holdout.

$$ \min_{\beta}\ \lVert y-X\beta\rVert_2^{2}+\alpha\lVert\beta\rVert_2^{2} $$

$$ \mathrm{MAD}=\frac{1}{n}\sum_{t=1}^{n}\left|y_t-\hat{y}_t\right| $$

$$ \mathrm{MSE}=\frac{1}{n}\sum_{t=1}^{n}\left(y_t-\hat{y}_t\right)^{2} $$

$$ \mathrm{RMSE}=\sqrt{\mathrm{MSE}} $$

$$ \mathrm{MAPE}=\frac{100}{|T_0|}\sum_{t\in T_0}\left|\frac{y_t-\hat{y}_t}{y_t}\right| $$

$$ \mathrm{MPE}=\frac{100}{|T_0|}\sum_{t\in T_0}\frac{y_t-\hat{y}_t}{y_t} $$

In [7]:
pipeline_result = run(
    PipelineConfig(input=input_path, outdir=reports_path, formats=("png",), seed=42)
)
metrics_table = pd.read_csv(reports_path / "tables/error_metrics.csv")
display(metrics_table[["produto", "modelo", "mad", "mape", "mse", "rmse", "mpe", "vencedor"]])
display(fig_metrics_bar(metrics_table))
display(fig_error_table(metrics_table))

,produto,modelo,mad,mape,mse,rmse,mpe,vencedor
0,doce_de_frutas,ridge,14.387370,34.554863,396.699586,19.917319,30.606316,True
1,doce_de_frutas,random_forest,11.762302,36.736363,203.447544,14.263504,-21.509928,False
2,doce_de_frutas,gradient_boosting,12.035722,42.052829,186.977548,13.673973,-26.388278,False
3,doce_de_frutas,linear_normal_equation,22.656594,71.962701,648.235994,25.460479,-36.087272,False
4,doce_de_frutas,exponential_loglinear,22.073870,63.335937,681.009310,26.096155,-22.680127,False
5,doce_de_frutas,exponential_normal_equation,22.073870,63.335937,681.009310,26.096155,-22.680127,False
6,geleia_de_frutas,ridge,9.014332,74.919980,124.385376,11.152819,-71.791438,False
7,geleia_de_frutas,random_forest,2.889991,18.697976,15.899182,3.987378,-6.213059,True
8,geleia_de_frutas,gradient_boosting,3.471987,25.413135,23.469980,4.844583,-14.611618,False
9,geleia_de_frutas,linear_normal_equation,8.515024,77.307328,87.483855,9.353280,-54.234199,False


<Figure size 1000x600 with 1 Axes>

<Figure size 1000x600 with 1 Axes>

## Backtest de 2024
O holdout é 2024, treinado somente com 2022–2023. A previsão recursiva usa
$$\hat{y}_{t+h}=f(\hat{y}_{t+h-1},y_{t+h-12},\mathrm{mês}_{t+h},\mathrm{trend}_{t+h},c_{m(t+h)}).$$
Assim, $\hat{y}_{t+h-1}$ é a previsão anterior e nenhum $y_t$ real de 2024 entra como `lag_1`.

$$ \hat{y}_{t+h}=f\!\left(\hat{y}_{t+h-1},\ y_{t+h-12},\ \text{mês}_{t+h},\ \text{trend}_{t+h},\ c_{m(t+h)}\right) $$

Os modelos clássicos (`linear_normal_equation`, `exponential_loglinear`, `exponential_normal_equation`) não dependem de `lag_1`/`lag_12`: são função apenas do índice de tendência $x$, então o holdout de 2024 é avaliado por predição direta $\hat{y}_t=\hat{y}(x_t)$, sem recursão. Cada um dos 6 métodos (`ridge`, `random_forest`, `gradient_boosting`, `linear_normal_equation`, `exponential_loglinear`, `exponential_normal_equation`) grava seu backtest por produto em PNG/SVG/PDF numa pasta própria: `reports/.../figures/{png,svg,pdf}/<método>/backtest_<produto>.<formato>`.

In [8]:
month_labels = month_clusters.labels
classical_regressions = {
    "linear_normal_equation": linear_regression_normal_equation,
    "exponential_loglinear": exponential_regression_loglinear,
    "exponential_normal_equation": exponential_regression_normal_equation,
}
backtest_predictions = {}
for product in products:
    product_frame = analysis_frame.loc[analysis_frame["produto"].eq(product)].sort_values("data")
    history = product_frame.set_index("data")["unidades"]
    product_features = build_features(product_frame, month_labels)
    target = history.reindex(product_features.index)
    train_x, train_y, _, _ = split_train_holdout(product_features, target, 2024)
    history_train = history.loc[history.index.year < 2024]
    backtest_predictions[product] = {}
    for model_name, estimator in build_estimators(seed=42).items():
        fitted = fit_final(estimator, train_x, train_y)
        backtest_predictions[product][model_name] = recursive_backtest(
            fitted, history_train, partial(next_row, month_clusters=month_labels), 12
        )
    ordered_frame = product_frame.reset_index(drop=True)
    x_trend = ordered_frame.index.to_numpy(dtype=float)
    y_trend = ordered_frame["unidades"].to_numpy(dtype=float)
    trend_years = ordered_frame["ano"].to_numpy()
    trend_train_mask = trend_years < 2024
    trend_holdout_mask = trend_years == 2024
    holdout_dates = pd.to_datetime(ordered_frame.loc[trend_holdout_mask, "data"])
    for model_name, builder in classical_regressions.items():
        classical_fit = builder(x_trend[trend_train_mask], y_trend[trend_train_mask])
        if not classical_fit.applicable:
            continue
        predicted = classical_fit.predict(x_trend[trend_holdout_mask])
        backtest_predictions[product][model_name] = pd.Series(
            predicted, index=holdout_dates, name=model_name
        )
display(fig_backtest(analysis_frame, backtest_predictions[products[0]], products[0]))

method_folders = tuple(backtest_predictions[products[0]].keys())
saved_method_plots: list[Path] = []
for product in products:
    for model_name in method_folders:
        series = backtest_predictions[product].get(model_name)
        if series is None:
            continue
        method_figure = fig_backtest(analysis_frame, {model_name: series}, product)
        saved_method_plots.extend(
            save_figure(method_figure, f"{model_name}/backtest_{product}", figures_dir)
        )
        plt.close(method_figure)
print(
    f"{len(saved_method_plots)} arquivos gravados em {figures_dir / 'png'} "
    f"(1 pasta por método: {', '.join(method_folders)})"
)

<Figure size 1000x600 with 1 Axes>

54 arquivos gravados em /Users/borinmacedo/GitRepos/insper-bebidem/reports/notebook-example/figures/png (1 pasta por método: ridge, random_forest, gradient_boosting, linear_normal_equation, exponential_loglinear, exponential_normal_equation)


## Previsão dos próximos anos
A previsão futura começa em 2025-01 e não reprevê o histórico. Para $h=1,2,3$ anos, a tabela anual soma os 12 valores mensais de cada ano. $\hat{y}_t$ é a demanda prevista e o total anual é $\sum_{m=1}^{12}\hat{y}_{a,m}$.

$$ \hat{y}_{t+h}=f\!\left(\hat{y}_{t+h-1},\ y_{t+h-12},\ \text{mês}_{t+h},\ \text{trend}_{t+h},\ c_{m(t+h)}\right) $$

In [9]:
forecast_monthly = pd.read_csv(reports_path / "tables/forecast_monthly.csv")
forecast_monthly["data"] = pd.to_datetime(
    forecast_monthly[["ano", "mes"]].assign(day=1).rename(columns={"ano": "year", "mes": "month"})
)
forecast_annual = pd.read_csv(reports_path / "tables/forecast_annual.csv")
display(forecast_annual)
best_models = metrics_table.loc[metrics_table["vencedor"]].set_index("produto")["modelo"].to_dict()
display(fig_forecast_all(analysis_frame, forecast_monthly, best_models, metrics_table))
forecast_figures = {
    product: fig_forecast_product(
        analysis_frame, forecast_monthly, product, holdout_metrics=metrics_table
    )
    for product in products
}
for figure in forecast_figures.values():
    display(figure)
    plt.close(figure)

,produto,ano,unidades_previstas_total,modelo,mape_holdout,mad_holdout
0,doce_de_frutas,2025,511,ridge,34.554863,14.387370
1,geleia_de_frutas,2025,180,random_forest,18.697976,2.889991
2,suco_premium,2025,814,gradient_boosting,9.922177,5.826697


<Figure size 1000x600 with 1 Axes>

<Figure size 1000x600 with 1 Axes>

<Figure size 1000x600 with 1 Axes>

<Figure size 1000x600 with 1 Axes>

## Métodos de séries temporais para ciclos periódicos
Para capturar os ciclos sazonais periódicos observados nos dados, comparamos três abordagens clássicas de séries temporais:

1. **Holt-Winters Multiplicativo**: suavização exponencial com nível, tendência e sazonalidade multiplicativa ($m=12$).
2. **SARIMA $(1,0,0)\times(1,0,0)_{12}$**: modelo autorregressivo sazonal.
3. **Fourier (Regressão Harmônica)**: regressão determinística com tendência linear e $K=2$ harmônicos.

### Resumo de escolha

| Método | Adequação a N=37 | Captura ciclo anual | Trata amplitude variável | Risco de overfitting |
| --- | --- | --- | --- | --- |
| **Holt-Winters Multiplicativo** | Alta | Sim (m=12) | Sim | Baixo |
| **SARIMA** | Baixa | Sim | Com $\log(y)$ | Alto (perde 12 pts) |
| **Fourier (Regressão Harmônica)** | Alta | Sim (P=12) | Não (amplitude fixa) | Muito baixo |

$$ \hat{y}_{t+h} = (\ell_t + h b_t)\, s_{t+h-12} $$

$$ y(t) = a_0 + b\,t + \sum_{k=1}^{K}\left[a_k\cos\!\left(\frac{2\pi k t}{P}\right) + b_k\sin\!\left(\frac{2\pi k t}{P}\right)\right] $$


In [10]:
from bebibem.timeseries import (
    fig_timeseries_comparison,
    fig_timeseries_error_table,
    fig_timeseries_forecast_all,
    fig_timeseries_metrics_bar,
    generate_and_save_timeseries_artifacts,
)

ts_artifacts = generate_and_save_timeseries_artifacts(
    analysis_frame,
    reports_path,
    formats=("png",),
    horizon_years=1,
)
ts_metrics = ts_artifacts["metrics"]
display(ts_metrics)
display(fig_timeseries_error_table(ts_metrics))
display(fig_timeseries_metrics_bar(ts_metrics))

for product in products:
    fig_comp = fig_timeseries_comparison(
        analysis_frame.loc[analysis_frame["produto"].eq(product)],
        ts_artifacts["models"][product],
        product,
    )
    display(fig_comp)
    plt.close(fig_comp)

display(fig_timeseries_forecast_all(analysis_frame, ts_artifacts["monthly_forecast"]))


/Users/borinmacedo/GitRepos/insper-bebidem/.venv/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/Users/borinmacedo/GitRepos/insper-bebidem/.venv/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/Users/borinmacedo/GitRepos/insper-bebidem/.venv/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/Users/borinmacedo/GitRepos/insper-bebidem/.venv/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/Users/borinmacedo/GitRepos/insper-b

,produto,modelo,mad,mse,rmse,mape,mpe,n_zeros,metric_note,vencedor
0,doce_de_frutas,holt_winters_multiplicative,6.096754,75.668356,8.698756,15.529689,-2.993146,0,,True
1,doce_de_frutas,sarima,11.758153,174.996495,13.228624,37.611311,-22.601120,0,,False
2,doce_de_frutas,fourier_harmonic,12.864126,299.942589,17.318851,46.037063,-23.079749,0,,False
3,geleia_de_frutas,holt_winters_multiplicative,3.563600,24.733548,4.973283,27.511575,-24.781435,0,,True
4,geleia_de_frutas,sarima,5.011731,48.537922,6.966916,35.358446,-30.123023,0,,False
5,geleia_de_frutas,fourier_harmonic,5.565863,41.805635,6.465728,50.843527,-39.834932,0,,False
6,suco_premium,holt_winters_multiplicative,6.293420,49.316796,7.022592,11.161327,-9.005228,0,,True
7,suco_premium,sarima,8.280539,85.859365,9.266033,16.899306,-16.899306,0,,False
8,suco_premium,fourier_harmonic,17.258905,457.628079,21.392243,34.965878,-26.227084,0,,False


<Figure size 1400x600 with 1 Axes>

<Figure size 1400x600 with 2 Axes>

/Users/borinmacedo/GitRepos/insper-bebidem/.venv/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/Users/borinmacedo/GitRepos/insper-bebidem/.venv/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/Users/borinmacedo/GitRepos/insper-bebidem/.venv/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)


...

/Users/borinmacedo/GitRepos/insper-bebidem/.venv/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/Users/borinmacedo/GitRepos/insper-bebidem/.venv/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/Users/borinmacedo/GitRepos/insper-bebidem/.venv/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)


<Figure size 1000x600 with 1 Axes>

/Users/borinmacedo/GitRepos/insper-bebidem/.venv/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/Users/borinmacedo/GitRepos/insper-bebidem/.venv/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/Users/borinmacedo/GitRepos/insper-bebidem/.venv/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)


<Figure size 1000x600 with 1 Axes>

<Figure size 1000x600 with 1 Axes>

## Referência das fórmulas
As fórmulas usadas nas figuras e nos artefatos são centralizadas em `bebibem.formulas.FORMULAS`; os símbolos estão em `bebibem.formulas.SYMBOLS`. O pipeline grava as mesmas tabelas e figuras em `reports_path` (isolado de `reports/`, a raiz de produção do CLI).

## Resultados e discussão — melhor modelo por linha e previsão 2025
Até aqui, os modelos de ML/regressão clássica (`error_metrics.csv`, 6 candidatos) e os modelos de séries temporais (`timeseries_error_metrics.csv`, 3 candidatos) foram comparados **em grupos separados** — cada grupo elege seu próprio vencedor por menor MAPE no holdout 2024 (empate < 0,5 p.p. desempatado por menor MAD, `bebibem.models.select_best`). Isso pode escolher um modelo pior globalmente: no caso de Doce de frutas, o vencedor do grupo ML (`ridge`, MAPE 34,55%) é bem mais fraco que o vencedor do grupo de séries temporais (`holt_winters_multiplicative`, MAPE 15,53%), mas os dois grupos nunca foram comparados entre si.

Nesta seção, `bebibem.models.select_overall_best` combina as duas tabelas de métricas (9 candidatos no total) e aplica a mesma regra de seleção uma única vez por produto, elegendo o verdadeiro melhor modelo entre todos os métodos avaliados. A previsão mensal de 2025 exibida abaixo vem sempre do modelo vencedor **geral** de cada linha — de `forecast_monthly.csv` quando o vencedor é um modelo de ML/regressão clássica, ou de `timeseries_forecast_monthly.csv` (recalculado por produto após a correção de `forecast_timeseries_future`, que antes fixava Holt-Winters para todas as linhas independentemente do vencedor) quando o vencedor é um método de série temporal. A tabela combinada dos 9 candidatos (`produto`, `modelo`, `grupo`, MAD, MSE, RMSE, MAPE, MPE, `vencedor_geral`) também é renderizada como figura por `bebibem.figures.fig_combined_error_table` e gravada em `reports_path/figures/{png,svg,pdf}/previsao/tabela_comparativa_modelos.<formato>`.

In [11]:
from bebibem.figures import fig_combined_error_table
from bebibem.models import select_overall_best

overall_best = select_overall_best(metrics_table, ts_metrics)

metric_columns = ["produto", "modelo", "mad", "mse", "rmse", "mape", "mpe"]
combined_metrics = pd.concat(
    [
        metrics_table[metric_columns].assign(grupo="ML/regressão clássica"),
        ts_metrics[metric_columns].assign(grupo="séries temporais"),
    ],
    ignore_index=True,
)
combined_metrics["vencedor_geral"] = [
    overall_best.get(row.produto) == row.modelo for row in combined_metrics.itertuples()
]
display(
    combined_metrics.sort_values(["produto", "mape"]).reset_index(drop=True)
)
display(
    pd.Series(overall_best, name="modelo_vencedor_geral")
    .rename_axis("produto")
    .reset_index()
)

combined_error_table_figure = fig_combined_error_table(combined_metrics)
display(combined_error_table_figure)
saved_combined_table_files = save_figure(
    combined_error_table_figure,
    "previsao/tabela_comparativa_modelos",
    reports_path / "figures",
)
plt.close(combined_error_table_figure)
print(f"{len(saved_combined_table_files)} arquivo(s) gravados em {reports_path / 'figures'}")

,produto,modelo,mad,mse,rmse,mape,mpe,grupo,vencedor_geral
0,doce_de_frutas,holt_winters_multiplicative,6.096754,75.668356,8.698756,15.529689,-2.993146,séries temporais,True
1,doce_de_frutas,ridge,14.387370,396.699586,19.917319,34.554863,30.606316,ML/regressão clássica,False
2,doce_de_frutas,random_forest,11.762302,203.447544,14.263504,36.736363,-21.509928,ML/regressão clássica,False
3,doce_de_frutas,sarima,11.758153,174.996495,13.228624,37.611311,-22.601120,séries temporais,False
4,doce_de_frutas,gradient_boosting,12.035722,186.977548,13.673973,42.052829,-26.388278,ML/regressão clássica,False
5,doce_de_frutas,fourier_harmonic,12.864126,299.942589,17.318851,46.037063,-23.079749,séries temporais,False
6,doce_de_frutas,exponential_normal_equation,22.073870,681.009310,26.096155,63.335937,-22.680127,ML/regressão clássica,False
7,doce_de_frutas,exponential_loglinear,22.073870,681.009310,26.096155,63.335937,-22.680127,ML/regressão clássica,False
8,doce_de_frutas,linear_normal_equation,22.656594,648.235994,25.460479,71.962701,-36.087272,ML/regressão clássica,False
9,geleia_de_frutas,random_forest,2.889991,15.899182,3.987378,18.697976,-6.213059,ML/regressão clássica,True


,produto,modelo_vencedor_geral
0,doce_de_frutas,holt_winters_multiplicative
1,geleia_de_frutas,random_forest
2,suco_premium,gradient_boosting


<Figure size 1000x600 with 1 Axes>

3 arquivo(s) gravados em /Users/borinmacedo/GitRepos/insper-bebidem/reports/notebook-example/figures


### Vendas por mês — observado vs. previsto (2022–2025)
O histograma abaixo compara, para cada produto e cada ano, as vendas realmente observadas com o que o modelo vencedor geral daquela linha (DEM-27) previu para aquele mesmo ano: ajuste no próprio treino para 2022–2023, previsão fora da amostra (holdout) para 2024, e previsão genuína para 2025. O eixo X usa o nome do mês (Janeiro, Fevereiro, ..., Dezembro); cada barra de ano traz seu total anual na legenda, e toda barra de previsão aparece com textura hachurada.

In [12]:
_ts_forecast_slice = ts_artifacts["monthly_forecast"][["produto", "ano", "mes", "unidades_previstas", "modelo"]]
_ml_forecast_slice = forecast_monthly[["produto", "ano", "mes", "unidades_previstas", "modelo"]]
_forecast_by_model = pd.concat([_ts_forecast_slice, _ml_forecast_slice], ignore_index=True)
historical_forecast_2025 = pd.concat(
    [
        _forecast_by_model.loc[
            _forecast_by_model["produto"].eq(product)
            & _forecast_by_model["modelo"].eq(overall_best[product])
        ]
        for product in products
    ],
    ignore_index=True,
)

_ml_estimators = build_estimators(seed=42)

historical_prediction_rows = []
for product in products:
    winning_model = overall_best[product]
    series = reconstruct_winning_series(
        product,
        winning_model,
        ts_models=ts_artifacts["models"].get(product, {}),
        analysis_frame=analysis_frame,
        month_cluster_labels=month_clusters.labels,
        ml_estimators=_ml_estimators,
    )
    historical_prediction_rows.append(
        pd.DataFrame(
            {
                "produto": product,
                "ano": series.index.year,
                "mes": series.index.month,
                "unidades_previstas": series.to_numpy(dtype=float),
                "modelo": winning_model,
            }
        )
    )

historical_forecast_full = pd.concat(
    [
        *historical_prediction_rows,
        historical_forecast_2025[["produto", "ano", "mes", "unidades_previstas", "modelo"]],
    ],
    ignore_index=True,
)

historical_figures_2022_2024 = {
    product: fig_historical_bar(analysis_frame, product, forecast_frame=historical_forecast_full)
    for product in products
}
for figure in historical_figures_2022_2024.values():
    display(figure)

saved_historical_files = []
for product, figure in historical_figures_2022_2024.items():
    saved_historical_files.extend(
        save_figure(figure, f"previsao/observado_2022_2024_{product}", reports_path / "figures")
    )
    plt.close(figure)
print(f"{len(saved_historical_files)} arquivo(s) gravados em {reports_path / 'figures'}")

<Figure size 1000x600 with 1 Axes>

<Figure size 1000x600 with 1 Axes>

<Figure size 1000x600 with 1 Axes>

9 arquivo(s) gravados em /Users/borinmacedo/GitRepos/insper-bebidem/reports/notebook-example/figures


### Séries temporais e previsão — saída do melhor modelo vs. observado (T=0..47)
Para cada produto, o gráfico abaixo usa o modelo vencedor geral (DEM-27) para reconstruir sua própria saída contínua ao longo de todo o eixo T — ajuste no treino (2022–2023), previsão fora da amostra no holdout (2024) e previsão genuína (2025), tudo em uma única linha contínua — e sobrepõe os pontos observados em preto, para visualizar exatamente onde e o quanto cada previsão se afasta do valor real. Arquivos salvos em `series_temporais/melhor_modelo_<produto>` e `previsao/melhor_modelo_<produto>`.

In [13]:
from bebibem.figures import fig_best_model_fit

melhor_modelo_figures = {
    product: fig_best_model_fit(
        analysis_frame.loc[analysis_frame["produto"].eq(product)],
        historical_forecast_full.loc[historical_forecast_full["produto"].eq(product)],
        product,
    )
    for product in products
}
for figure in melhor_modelo_figures.values():
    display(figure)

saved_best_model_files = []
for product, figure in melhor_modelo_figures.items():
    for folder in ("series_temporais", "previsao"):
        saved_best_model_files.extend(
            save_figure(figure, f"{folder}/melhor_modelo_{product}", reports_path / "figures")
        )
    plt.close(figure)
print(f"{len(saved_best_model_files)} arquivo(s) gravados em {reports_path / 'figures'}")

<Figure size 1000x600 with 1 Axes>

<Figure size 1000x600 with 1 Axes>

<Figure size 1000x600 with 1 Axes>

18 arquivo(s) gravados em /Users/borinmacedo/GitRepos/insper-bebidem/reports/notebook-example/figures


### Previsão 2025 — barras isoladas, curva com histórico sobreposto
Os três gráficos de barra abaixo (por produto e o total somado) mostram **apenas** os valores previstos para 2025 (modelo vencedor geral de cada linha, DEM-27), sem pontos observados sobrepostos: misturar um valor de barra (previsão) com um ponto disperso de histórico no mesmo eixo mistura duas geometrias de gráfico incompatíveis e não ajuda a leitura — a comparação observado-vs-previsto por ano já é feita por `fig_historical_bar` (seção anterior, "Vendas por mês"). O eixo usa o índice inteiro contínuo $T=12(\text{ano}-2022)+(\text{mês}-1)$ definido acima, restrito a `T=36`..`T=47` (os 12 meses de 2025), valor exato por barra e total no título. Em seguida, um gráfico de **linhas** com as três curvas de histórico (2022–2024, pontos sobrepostos) + previsão 2025 sobrepostas (uma por produto, rotulada com o modelo vencedor geral e o MAPE de holdout de 2024) — aqui a mistura faz sentido porque histórico e previsão usam a mesma geometria de linha contínua, no mesmo padrão já usado por `fig_best_model_fit` (DEM-27) e `fig_backtest` (DEM-10, corrigida para mostrar 2022–2024 completo em vez de só o holdout 2024).

In [14]:
product_label_map = analysis_frame.drop_duplicates("produto").set_index("produto")["produto_label"]
timeseries_models = set(ts_metrics["modelo"])

final_forecast_rows = []
for product in products:
    winning_model = overall_best[product]
    source = (
        ts_artifacts["monthly_forecast"]
        if winning_model in timeseries_models
        else forecast_monthly
    )
    subset = source.loc[
        source["produto"].eq(product) & source["modelo"].eq(winning_model),
        ["produto", "ano", "mes", "unidades_previstas", "modelo"],
    ].copy()
    final_forecast_rows.append(subset)

final_forecast_2025 = (
    pd.concat(final_forecast_rows, ignore_index=True)
    .sort_values(["produto", "mes"])
    .reset_index(drop=True)
)
final_forecast_2025["data"] = pd.to_datetime(
    final_forecast_2025[["ano", "mes"]].assign(day=1).rename(columns={"ano": "year", "mes": "month"})
)
final_forecast_2025["produto_label"] = final_forecast_2025["produto"].map(product_label_map)
display(final_forecast_2025[["produto_label", "mes", "data", "modelo", "unidades_previstas"]])

annual_totals_2025 = (
    final_forecast_2025.groupby(["produto", "modelo"])["unidades_previstas"]
    .sum()
    .round(0)
    .astype(int)
    .rename("unidades_previstas_total")
    .reset_index()
)
annual_totals_2025["produto_label"] = annual_totals_2025["produto"].map(product_label_map)
display(annual_totals_2025[["produto_label", "modelo", "unidades_previstas_total"]])

,produto_label,mes,data,modelo,unidades_previstas
0,Doce de frutas,1,2025-01-01,holt_winters_multiplicative,70.287362
1,Doce de frutas,2,2025-02-01,holt_winters_multiplicative,55.307881
2,Doce de frutas,3,2025-03-01,holt_winters_multiplicative,22.479865
3,Doce de frutas,4,2025-04-01,holt_winters_multiplicative,33.693509
4,Doce de frutas,5,2025-05-01,holt_winters_multiplicative,89.400737
5,Doce de frutas,6,2025-06-01,holt_winters_multiplicative,20.279927
6,Doce de frutas,7,2025-07-01,holt_winters_multiplicative,19.420160
7,Doce de frutas,8,2025-08-01,holt_winters_multiplicative,19.234512
8,Doce de frutas,9,2025-09-01,holt_winters_multiplicative,21.904379
9,Doce de frutas,10,2025-10-01,holt_winters_multiplicative,30.619540


,produto_label,modelo,unidades_previstas_total
0,Doce de frutas,holt_winters_multiplicative,523
1,Geleia de frutas,random_forest,180
2,Suco Premium,gradient_boosting,814


In [15]:
product_figures_2025 = {
    product: fig_forecast_2025_bar(final_forecast_2025, product, overall_best)
    for product in products
}
for figure in product_figures_2025.values():
    display(figure)

total_figure_2025 = fig_forecast_2025_total_bar(final_forecast_2025)
display(total_figure_2025)

curves_figure_2025 = fig_forecast_2025_curves_all(
    final_forecast_2025, combined_metrics, history=analysis_frame
)
display(curves_figure_2025)

saved_result_files = []
for product, figure in product_figures_2025.items():
    saved_result_files.extend(
        save_figure(figure, f"previsao/previsao_2025_{product}", reports_path / "figures")
    )
    plt.close(figure)
saved_result_files.extend(
    save_figure(total_figure_2025, "previsao/previsao_2025_total", reports_path / "figures")
)
plt.close(total_figure_2025)
saved_result_files.extend(
    save_figure(curves_figure_2025, "previsao/previsao_2025_curvas", reports_path / "figures")
)
plt.close(curves_figure_2025)
print(f"{len(saved_result_files)} arquivo(s) gravados em {reports_path / 'figures'}")

<Figure size 1000x600 with 1 Axes>

<Figure size 1000x600 with 1 Axes>

<Figure size 1000x600 with 1 Axes>

<Figure size 1000x600 with 1 Axes>

<Figure size 1000x600 with 1 Axes>

15 arquivo(s) gravados em /Users/borinmacedo/GitRepos/insper-bebidem/reports/notebook-example/figures


In [16]:
from IPython.display import Markdown

discussion_lines = ["### Discussão\n"]
for product in products:
    winning_model = overall_best[product]
    mape_value = combined_metrics.loc[
        combined_metrics["produto"].eq(product) & combined_metrics["modelo"].eq(winning_model), "mape"
    ].iloc[0]
    total_2025 = int(
        annual_totals_2025.loc[
            annual_totals_2025["produto"].eq(product) & annual_totals_2025["modelo"].eq(winning_model),
            "unidades_previstas_total",
        ].iloc[0]
    )
    silo_group = "séries temporais" if winning_model in timeseries_models else "ML/regressão clássica"
    discussion_lines.append(
        f"- **{product_label_map[product]}**: vencedor geral `{winning_model}` "
        f"(grupo {silo_group}), MAPE holdout 2024 = {mape_value:.2f}%, "
        f"total previsto para 2025 = {total_2025} unidades."
    )

ridge_mape = combined_metrics.loc[
    combined_metrics["produto"].eq("doce_de_frutas") & combined_metrics["modelo"].eq("ridge"), "mape"
].iloc[0]
hw_mape = combined_metrics.loc[
    combined_metrics["produto"].eq("doce_de_frutas") & combined_metrics["modelo"].eq("holt_winters_multiplicative"),
    "mape",
].iloc[0]
discussion_lines.append(
    "\nA comparação unificada muda o modelo de produção de **Doce de frutas**: o antigo vencedor "
    f"isolado do grupo ML (`ridge`, MAPE {ridge_mape:.2f}%) perde para `holt_winters_multiplicative` "
    f"(MAPE {hw_mape:.2f}%) do grupo de séries temporais — {ridge_mape / hw_mape:.1f}x menor erro. "
    "Suco Premium e Geleia de frutas mantêm seus vencedores originais (`gradient_boosting` e "
    "`random_forest`), que já batiam os três métodos de série temporal nesses dois produtos."
)
display(Markdown("\n".join(discussion_lines)))

### Discussão

- **Doce de frutas**: vencedor geral `holt_winters_multiplicative` (grupo séries temporais), MAPE holdout 2024 = 15.53%, total previsto para 2025 = 523 unidades.
- **Geleia de frutas**: vencedor geral `random_forest` (grupo ML/regressão clássica), MAPE holdout 2024 = 18.70%, total previsto para 2025 = 180 unidades.
- **Suco Premium**: vencedor geral `gradient_boosting` (grupo ML/regressão clássica), MAPE holdout 2024 = 9.92%, total previsto para 2025 = 814 unidades.

A comparação unificada muda o modelo de produção de **Doce de frutas**: o antigo vencedor isolado do grupo ML (`ridge`, MAPE 34.55%) perde para `holt_winters_multiplicative` (MAPE 15.53%) do grupo de séries temporais — 2.2x menor erro. Suco Premium e Geleia de frutas mantêm seus vencedores originais (`gradient_boosting` e `random_forest`), que já batiam os três métodos de série temporal nesses dois produtos.

### Resumo final — previsão total 2025 por produto e total geral
Fecha a análise com o número de negócio isolado: o total anual previsto para 2025 de cada linha (modelo vencedor geral, DEM-27) e a soma das três linhas.

In [17]:
from bebibem.figures import fig_forecast_2025_summary_table

resumo_previsao_2025 = annual_totals_2025.loc[
    annual_totals_2025.apply(lambda row: overall_best.get(row["produto"]) == row["modelo"], axis=1)
][["produto_label", "modelo", "unidades_previstas_total"]].reset_index(drop=True)

total_geral_2025 = int(resumo_previsao_2025["unidades_previstas_total"].sum())
resumo_previsao_2025.loc[len(resumo_previsao_2025)] = ["Total (3 linhas)", "—", total_geral_2025]

display(resumo_previsao_2025)

summary_table_figure = fig_forecast_2025_summary_table(resumo_previsao_2025)
display(summary_table_figure)
saved_summary_table_files = save_figure(
    summary_table_figure, "previsao/previsao_2025_resumo_total", reports_path / "figures"
)
plt.close(summary_table_figure)
print(f"{len(saved_summary_table_files)} arquivo(s) gravados em {reports_path / 'figures'}")

,produto_label,modelo,unidades_previstas_total
0,Doce de frutas,holt_winters_multiplicative,523
1,Geleia de frutas,random_forest,180
2,Suco Premium,gradient_boosting,814
3,Total (3 linhas),—,1517


<Figure size 1000x600 with 1 Axes>

3 arquivo(s) gravados em /Users/borinmacedo/GitRepos/insper-bebidem/reports/notebook-example/figures
